In [ ]:
from pathlib import Path
import ast
import html
import re
import unicodedata

import pandas as pd

REVIEWS_FILENAME = "test_data.csv"
PRODUCTS_FILENAME = "title_brand.csv"
OUTPUT_FILENAME = "sentiment_ready_dataset.csv"


def locate_input(filename: str) -> Path:
    """Find an input beside the notebook/current directory or in /mnt/data."""
    candidates = [Path.cwd() / filename, Path('/mnt/data') / filename]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"فایل {filename} پیدا نشد. آن را کنار نوت‌بوک قرار دهید یا مسیر ورودی را اصلاح کنید."
    )


reviews_path = locate_input(REVIEWS_FILENAME)
products_path = locate_input(PRODUCTS_FILENAME)
output_path = reviews_path.parent / OUTPUT_FILENAME

print("Reviews:", reviews_path)
print("Products:", products_path)
print("Output:", output_path)


In [ ]:
reviews_raw = pd.read_csv(
    reviews_path,
    encoding="utf-8-sig",
    dtype={"asin": "string", "reviewerID": "string"},
    low_memory=False,
)
products_raw = pd.read_csv(
    products_path,
    encoding="utf-8-sig",
    dtype={"asin": "string"},
    low_memory=False,
)

required_review_columns = {"asin", "reviewText", "summary", "reviewTime", "verified", "vote", "style"}
required_product_columns = {"asin", "title", "brand"}
missing_review_columns = required_review_columns - set(reviews_raw.columns)
missing_product_columns = required_product_columns - set(products_raw.columns)
if missing_review_columns:
    raise ValueError(f"ستون‌های لازم در فایل نظرات وجود ندارند: {sorted(missing_review_columns)}")
if missing_product_columns:
    raise ValueError(f"ستون‌های لازم در فایل محصولات وجود ندارند: {sorted(missing_product_columns)}")

print(f"تعداد ردیف‌های خام نظرات: {len(reviews_raw):,}")
print(f"تعداد ردیف‌های خام محصولات: {len(products_raw):,}")
print("ستون‌های نظرات:", list(reviews_raw.columns))
print("ستون‌های محصولات:", list(products_raw.columns))


In [ ]:
def normalize_asin(values: pd.Series) -> pd.Series:
    """Trim whitespace and normalize product identifiers to uppercase."""
    return values.astype("string").str.strip().str.upper()


def clean_text(value) -> str:
    """Conservative English-review cleaning that preserves sentiment signals."""
    if value is None or pd.isna(value):
        return ""

    text = unicodedata.normalize("NFKC", str(value))
    text = html.unescape(text)
    text = re.sub(r"(?i)https?://\S+|www\.\S+", " URL ", text)
    text = re.sub(r"(?is)<(script|style).*?>.*?</\1>", " ", text)
    text = re.sub(r"(?s)<[^>]+>", " ", text)
    text = re.sub(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def first_nonempty(series: pd.Series):
    """Select the most frequent non-empty product value deterministically."""
    values = series.dropna().astype(str).str.strip()
    values = values[values.ne("")]
    if values.empty:
        return pd.NA
    counts = values.value_counts()
    return sorted(counts[counts == counts.max()].index)[0]


def format_style(value) -> str:
    """Convert dictionary-like style metadata into readable key/value text."""
    if value is None or pd.isna(value) or not str(value).strip():
        return ""
    raw = str(value).strip()
    try:
        parsed = ast.literal_eval(raw)  # Safe literal parser; never use eval().
        if isinstance(parsed, dict):
            parts = []
            for key, item in parsed.items():
                key_text = str(key).strip().rstrip(":")
                item_text = clean_text(item)
                if item_text:
                    parts.append(f"{key_text}: {item_text}" if key_text else item_text)
            return "; ".join(parts)
    except (ValueError, SyntaxError):
        pass
    return clean_text(raw)


# Copy input frames to avoid changing the original data in memory.
reviews = reviews_raw.copy()
products = products_raw.copy()

reviews["asin_key"] = normalize_asin(reviews["asin"])
products["asin_key"] = normalize_asin(products["asin"])

# Remove fully duplicated review rows only.
rows_before_dedup = len(reviews)
reviews = reviews.drop_duplicates().copy()
exact_review_duplicates_removed = rows_before_dedup - len(reviews)
assert not reviews.duplicated().any()

# The source product file contains repeated exact rows for some ASINs.
# One row per ASIN is sufficient for the many-reviews-to-one-product merge.
# Title and brand are only whitespace-trimmed here; heavy review-text cleaning
# is applied to the actual review and summary fields below.
products["title"] = products["title"].astype("string").str.strip()
products["brand"] = products["brand"].astype("string").str.strip()
product_lookup = products.drop_duplicates(subset=["asin_key"])[
    ["asin_key", "title", "brand"]
].rename(columns={"title": "product_title"})

# Clean textual fields while retaining punctuation, casing, and negations.
reviews["review_text_clean"] = reviews["reviewText"].map(clean_text)
reviews["summary_clean"] = reviews["summary"].map(clean_text)
reviews["style_attributes"] = reviews["style"].map(format_style)

# Create one model-input text field, including the short summary when available.
def combine_text(row) -> str:
    summary = row["summary_clean"]
    body = row["review_text_clean"]
    if summary and body:
        return f"{summary}. {body}"
    return summary or body

reviews["text_for_model"] = reviews.apply(combine_text, axis=1)

# Parse review date and numeric helpful-vote count.
reviews["review_date"] = pd.to_datetime(reviews["reviewTime"], errors="coerce").dt.strftime("%Y-%m-%d")
reviews["helpful_votes"] = pd.to_numeric(reviews["vote"], errors="coerce")
if reviews["helpful_votes"].dropna().mod(1).eq(0).all():
    reviews["helpful_votes"] = reviews["helpful_votes"].astype("Int64")

# Normalize verified-purchase values to nullable booleans.
verified_map = {True: True, False: False, "true": True, "false": False, "True": True, "False": False}
reviews["verified_purchase"] = reviews["verified"].map(verified_map).astype("boolean")

# Many reviews can map to the same product; each product occurs once in the lookup.
merged = reviews.merge(
    product_lookup,
    on="asin_key",
    how="left",
    validate="many_to_one",
    indicator=True,
)
product_match_rows = int(merged["_merge"].eq("both").sum())
product_unmatched_rows = int(merged["_merge"].eq("left_only").sum())

# Export useful model input and non-identifying review/product metadata.
final_columns = [
    ("asin_key", "asin"),
    ("brand", "brand"),
    ("product_title", "product_title"),
    ("text_for_model", "text_for_model"),
    ("review_text_clean", "review_text_clean"),
    ("summary_clean", "summary_clean"),
    ("review_date", "review_date"),
    ("verified_purchase", "verified_purchase"),
    ("helpful_votes", "helpful_votes"),
    ("style_attributes", "style_attributes"),
]
final_df = merged[[source for source, _ in final_columns]].copy()
final_df.columns = [target for _, target in final_columns]

print(f"ردیف‌های تکراری کامل حذف‌شده از نظرات: {exact_review_duplicates_removed:,}")
print(f"ردیف‌های متصل‌شده به محصول: {product_match_rows:,}")
print(f"ردیف‌های بدون تطبیق ASIN: {product_unmatched_rows:,}")
print(f"ردیف‌های فاقد نام برند پس از اتصال: {int(final_df['brand'].isna().sum()):,}")
print(f"ردیف‌های فاقد عنوان محصول پس از اتصال: {int(final_df['product_title'].isna().sum()):,}")


In [ ]:
assert final_df["asin"].notna().all()
assert final_df["text_for_model"].fillna("").str.strip().ne("").all(), "متن ورودی مدل خالی وجود دارد."
assert len(final_df) == len(reviews), "تعداد ردیف‌ها بعد از اتصال تغییر غیرمنتظره داشته است."

quality_report = pd.Series({
    "rows": len(final_df),
    "unique_products": final_df["asin"].nunique(),
    "missing_brand_rows": int(final_df["brand"].isna().sum()),
    "missing_product_title_rows": int(final_df["product_title"].isna().sum()),
    "missing_summary_rows": int(final_df["summary_clean"].eq("").sum()),
    "missing_helpful_vote_rows": int(final_df["helpful_votes"].isna().sum()),
    "empty_model_text_rows": int(final_df["text_for_model"].fillna("").str.strip().eq("").sum()),
})
print("گزارش کیفیت دیتاست:")
display(quality_report.to_frame("value"))
print("\nنمونه خروجی:")
sample_preview = final_df[["asin", "brand", "review_date", "text_for_model"]].head(3).copy()
sample_preview["text_for_model"] = sample_preview["text_for_model"].str.slice(0, 180)
display(sample_preview)


In [ ]:
final_df.to_csv(output_path, index=False, encoding="utf-8-sig", na_rep="")
print(f"\nفایل نهایی ذخیره شد: {output_path}")
print(f"ابعاد دیتاست نهایی: {final_df.shape[0]:,} ردیف × {final_df.shape[1]} ستون")
